# Step 4 — A1/A2/A3 GPT refinement + A4 random ablation (resumable)

Built from the supplied **original Step 4**. A1 and A3 use its unchanged GPT prompt, model request, scoring defaults, score formula and per-stance ranking. A2 uses the documented no-stance adaptation. A4 performs random balanced sampling without GPT.

**A1 small-pool rule:** keep `min(150, available)` per stance, including zero, without an exception or artificial replacement.

**A2/A3 inputs:** actual `*_A2_selected_400_step3.csv` and `*_A3_selected_400_step3.csv` from the FULL Step 3 archives. Paths below match their actual archive names.

Batch checkpoints and results are saved in Google Drive; TEST and FULL are isolated. Restart the notebook after a Colab disconnect using the same settings and source files.

In [ ]:
!pip -q install openai pandas tqdm

In [ ]:
import os, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from getpass import getpass
from google.colab import drive
from openai import AzureOpenAI

In [ ]:
# ===== Configuration =====
# Put the extracted A2/ and A3/ folders in BASE_DIR/inputs/.
# Example: inputs/A2/blm_A2_selected_400_step3.csv
#          inputs/A3/blm_A3_selected_400_step3.csv
# Put the three baseline selected_400_balanced_diverse files in inputs/A4/.
BASE_DIR = Path('/content')
RUN_MODE = 'FULL'                 # TEST or FULL
RUN_VARIANTS = ['A1']   # Use ['A1','A2','A3','A4'] to start all
TEST_N_PER_STANCE = 8
TEST_N_A2 = 20
BATCH_SIZE = 25
FINAL_PER_STANCE = 150
FINAL_PER_TOPIC = 300
RANDOM_SEED = 42
TEXT_COL = 'comment_text'
TOPICS = ['BLM','guns','same_sex_marriage']
AZURE_DEPLOYMENT_NAME = os.environ.get("AZURE_OPENAI_DEPLOYMENT")
TOPIC_STEMS = {'BLM':'blm','guns':'guns','same_sex_marriage':'samesex'}
INPUT_FILES = {
    'A1': {'BLM': BASE_DIR/'stage3/TEST/A1/blm_A1_selected_248_step3.csv', 'guns':BASE_DIR/'stage3/TEST/A1/guns_A1_selected_83_step3.csv', 'same_sex_marriage':BASE_DIR/'stage3/TEST/A1/samesex_A1_selected_43_step3.csv'},
    'A2': {t: BASE_DIR/'stage3'/'FULL'/'A2'/f'{TOPIC_STEMS[t]}_A2_selected_400_step3.csv' for t in TOPICS},
    'A3': {t: BASE_DIR/'stage3'/'FULL'/'A3'/f'{TOPIC_STEMS[t]}_A3_selected_400_step3.csv' for t in TOPICS},
    'A4': {t: BASE_DIR/'baseline_results'/f'{t}_selected_400_balanced_diverse.csv' for t in TOPICS},
}
OUTPUT_ROOT = BASE_DIR/'stage4'/RUN_MODE
CHECKPOINT_ROOT = BASE_DIR/'stage4'/'checkpoints'/RUN_MODE
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True,exist_ok=True)
print('Mode:',RUN_MODE,'Variants:',RUN_VARIANTS)
for v in RUN_VARIANTS:
    for t in TOPICS:
        print(v,t,'->',INPUT_FILES[v][t])

In [ ]:
# GPT client is needed only for A1, A2 and A3.
from getpass import getpass
from openai import AzureOpenAI
client = None
if any(v in RUN_VARIANTS for v in ('A1','A2','A3')):
  AZURE_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT") or os.environ.get("ENDPOINT_URL")
  AZURE_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
  if not AZURE_ENDPOINT or not AZURE_API_KEY:
      raise EnvironmentError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY before running.")
  client = AzureOpenAI(
      azure_endpoint=AZURE_ENDPOINT,
      api_key=AZURE_API_KEY,
      api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
  )

In [ ]:
def topic_description(topic_name):
    if topic_name == "BLM":
        return "Black Lives Matter"
    if topic_name == "guns":
        return "Gun control"
    if topic_name == "same_sex_marriage":
        return "Same-sex marriage legalization"
    return topic_name

In [ ]:
def build_gpt_refinement_prompt(topic_name, stance, batch_df, text_col):
    rows = []

    for _, row in batch_df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        text = text.replace("\n", " ").strip()

        rows.append({
            "id": int(row["_gpt_local_id"]),
            "stance_label": str(row["stance_label"]),
            "text": text[:1500],
        })

    return f"""
You are helping refine a research dataset of political social-media comments.

Topic:
{topic_description(topic_name)}

Current stance group:
{stance}

Goal:
Select the best comments for a final dataset that is diverse, representative, and useful for building opposite-opinion personas.

All comments in this batch are already relevant to the topic and already classified as {stance}.
Your task is not to change the stance label.
Your task is to score how useful each comment is for the final selected set.

Evaluate each comment using:

1. uniqueness_score:
1-5, where 5 means this comment expresses a distinct or rare viewpoint within the {stance} side.

2. coverage_score:
1-5, where 5 means this comment represents an important recurring viewpoint within the {stance} side.

3. redundancy_risk:
1-5, where 5 means this comment is likely repetitive or similar to many others.

4. clarity_score:
1-5, where 5 means the stance is clear and understandable.

5. persona_usefulness:
1-5, where 5 means this comment would be useful as background for a realistic opposite-opinion persona.

6. keep_score:
1-10, where 10 means this comment should strongly be kept in the final dataset.

Important rules:
Do not prefer polite or moderate comments just because they are safer.
Preserve extreme, emotional, or unpopular comments if they represent a distinct stance.
Do not penalize toxicity unless the comment is too unclear to be useful.
Focus on stance diversity and coverage within the {stance} side.

Return valid JSON only in this exact format:

{{
  "items": [
    {{
      "id": 0,
      "uniqueness_score": 4,
      "coverage_score": 5,
      "redundancy_risk": 2,
      "clarity_score": 5,
      "persona_usefulness": 5,
      "keep_score": 9,
      "short_reason": "brief reason"
    }}
  ]
}}

Comments:
{json.dumps(rows, ensure_ascii=False)}
"""

In [ ]:
def extract_json_from_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise

In [ ]:
def score_batch_with_gpt(topic_name, stance, batch_df, text_col, max_retries=4):
    prompt = build_gpt_refinement_prompt(topic_name, stance, batch_df, text_col)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a careful research assistant. Return valid JSON only."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            content = response.choices[0].message.content
            parsed = extract_json_from_text(content)

            return parsed.get("items", [])

        except Exception as e:
            print(f"Attempt {attempt + 1} failed:", repr(e))
            time.sleep(5 + attempt * 5)

    return []

## A2: prompt adapted to no stance classification

The six scores, scales, JSON structure, Azure call, and scoring formula are preserved. The A2 prompt does not falsely state that comments are already relevant or stance-labeled. A2 chooses the 300 highest-scoring comments overall, without a pro/anti quota.

In [ ]:
def build_a2_prompt(topic_name, batch_df, text_col):
    rows=[]
    for _,row in batch_df.iterrows():
        txt=str(row[text_col]) if pd.notna(row[text_col]) else ''
        rows.append({'id':int(row['_gpt_local_id']),'text':txt.replace('\n',' ').strip()[:1500]})
    return f"""You are helping refine a research dataset of political social-media comments.

Topic:
{topic_description(topic_name)}

Goal:
Select the best comments for a final dataset that is diverse, representative, and useful for building opposite-opinion personas.

These comments have NOT been stance-filtered or verified as relevant. Do not assume any stance label.
Evaluate their usefulness for this topic and for creating realistic opinion-based personas.

Evaluate each comment using:
1. uniqueness_score: 1-5; 5 means a distinct or rare viewpoint relevant to the topic.
2. coverage_score: 1-5; 5 means an important recurring viewpoint relevant to the topic.
3. redundancy_risk: 1-5; 5 means repetitive or similar to many others.
4. clarity_score: 1-5; 5 means clear and understandable.
5. persona_usefulness: 1-5; 5 means useful background for a realistic opinion persona on the topic.
6. keep_score: 1-10; 10 means strongly keep for the final dataset.

Important rules:
Comments unrelated to the topic or unusable for a topic-specific persona should receive low keep_score and persona_usefulness.
Do not prefer polite or moderate comments merely because they are safer.
Do not penalize toxicity unless it makes the comment too unclear to be useful.

Return valid JSON only in this exact format:

{{
  "items": [
    {{
      "id": 0,
      "uniqueness_score": 4,
      "coverage_score": 5,
      "redundancy_risk": 2,
      "clarity_score": 5,
      "persona_usefulness": 5,
      "keep_score": 9,
      "short_reason": "brief reason"
    }}
  ]
}}

Comments:
{json.dumps(rows, ensure_ascii=False)}
"""

def score_batch_a2(topic_name,batch_df,max_retries=4):
    prompt=build_a2_prompt(topic_name,batch_df,TEXT_COL)
    for attempt in range(max_retries):
        try:
            response=client.chat.completions.create(model=AZURE_DEPLOYMENT_NAME,messages=[
                {'role':'system','content':'You are a careful research assistant. Return valid JSON only.'},
                {'role':'user','content':prompt}])
            return extract_json_from_text(response.choices[0].message.content).get('items',[])
        except Exception as e:
            print(f'Attempt {attempt+1} failed:',repr(e));time.sleep(5+attempt*5)
    return []

In [ ]:
def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp')
    with open(temp,'w',encoding='utf-8') as f:json.dump(obj,f,ensure_ascii=False,indent=2)
    os.replace(temp,path)

def fingerprint(df,variant,topic):
    h=hashlib.sha256()
    cfg={'variant':variant,'topic':topic,'mode':RUN_MODE,'batch_size':BATCH_SIZE,
         'model':AZURE_DEPLOYMENT_NAME,'prompt_version':'baseline_step4_exact' if variant in ('A1','A3') else 'a2_no_stance_v1',
         'random_seed':RANDOM_SEED,'final_per_stance':FINAL_PER_STANCE,'final_per_topic':FINAL_PER_TOPIC}
    h.update(json.dumps(cfg,sort_keys=True).encode())
    for record in df.to_dict('records'):
        h.update(json.dumps(record,sort_keys=True,ensure_ascii=False,default=str).encode());h.update(b'\n')
    return h.hexdigest()

def read_input(variant,topic):
    path=Path(INPUT_FILES[variant][topic])
    if not path.exists():raise FileNotFoundError(f'Missing {variant}/{topic}: {path}. Update INPUT_FILES in the configuration cell.')
    df=pd.read_csv(path,encoding='utf-8-sig')
    if TEXT_COL not in df:raise ValueError(f'{path}: missing {TEXT_COL}')
    if variant!='A2':
        if 'stance_label' not in df:raise ValueError(f'{path}: missing stance_label')
        df['stance_label']=df['stance_label'].astype(str).str.lower().str.strip()
        df=df[df['stance_label'].isin(['pro','anti'])].copy().reset_index(drop=True)
        if RUN_MODE=='TEST':df=pd.concat([df[df.stance_label==s].head(TEST_N_PER_STANCE) for s in ('pro','anti')],ignore_index=True)
    else:
        df=df.reset_index(drop=True)
        if RUN_MODE=='TEST':df=df.head(TEST_N_A2).copy()
    if 'comment_id' in df and df.comment_id.duplicated().any():
        print(f'WARNING {variant}/{topic}: {df.comment_id.duplicated().sum()} duplicate comment_id values')
    return df.reset_index(drop=True)

def gpt_items_with_checkpoints(df,variant,topic,digest):
    df=df.copy();df['_gpt_local_id']=np.arange(len(df))
    all_items=[]
    for stance in (['pro','anti'] if variant!='A2' else ['all']):
        subset=df[df.stance_label==stance].copy() if stance!='all' else df.copy()
        for start in tqdm(range(0,len(subset),BATCH_SIZE),desc=f'{variant} {topic} {stance}'):
            batch=subset.iloc[start:start+BATCH_SIZE]
            path=CHECKPOINT_ROOT/variant/topic/stance/f'batch_{start:05d}.json'
            items=None
            if path.exists():
                with open(path,encoding='utf-8') as f: ck=json.load(f)
                if (ck.get('digest'),ck.get('start'),ck.get('size'))!=(digest,start,len(batch)):
                    raise ValueError(f'Checkpoint mismatch: {path}. Do not reuse checkpoints with changed inputs.')
                items=ck['items']
            if items is None:
                items=score_batch_a2(topic,batch) if variant=='A2' else score_batch_with_gpt(topic,stance,batch,TEXT_COL)
                atomic_json(path,{'digest':digest,'start':start,'size':len(batch),'items':items})
                time.sleep(0.5)
            all_items.extend(items)
    return df,all_items

In [ ]:
# Original Step 4 postprocessing preserved for A1/A3, with A2 selecting top 300 overall.
NUMERIC_COLS=['uniqueness_score','coverage_score','redundancy_risk','clarity_score','persona_usefulness','keep_score']
DEFAULTS={'id':None,'uniqueness_score':3,'coverage_score':3,'redundancy_risk':3,
          'clarity_score':3,'persona_usefulness':3,'keep_score':5,'short_reason':'missing_or_parse_error'}

def merge_scores(df,all_items):
    scores_df=pd.DataFrame(all_items)
    if scores_df.empty:scores_df=pd.DataFrame(columns=list(DEFAULTS))
    rename_map={'reason':'short_reason','rationale':'short_reason'}
    scores_df=scores_df.rename(columns={k:v for k,v in rename_map.items() if k in scores_df.columns})
    for col,default in DEFAULTS.items():
        if col not in scores_df:scores_df[col]=default
    scores_df['id']=pd.to_numeric(scores_df['id'],errors='coerce')
    scores_df=scores_df.dropna(subset=['id']);scores_df['id']=scores_df['id'].astype(int)
    for col in NUMERIC_COLS:
        scores_df[col]=pd.to_numeric(scores_df[col],errors='coerce').fillna(DEFAULTS[col])
    df_clean=df.drop(columns=['id','uniqueness_score','coverage_score','redundancy_risk','clarity_score',
                              'persona_usefulness','keep_score','short_reason','gpt_final_score'],errors='ignore')
    merged=df_clean.merge(scores_df[list(DEFAULTS)],left_on='_gpt_local_id',right_on='id',how='left')
    for col in NUMERIC_COLS:merged[col]=pd.to_numeric(merged[col],errors='coerce').fillna(DEFAULTS[col])
    merged['short_reason']=merged['short_reason'].fillna('missing_or_parse_error')
    merged['gpt_final_score']=(merged.keep_score+0.9*merged.coverage_score+0.9*merged.uniqueness_score
                               +0.6*merged.persona_usefulness+0.4*merged.clarity_score-0.8*merged.redundancy_risk)
    return merged

def finalize_gpt(merged,variant,topic):
    if variant=='A2':
        if RUN_MODE=='FULL' and len(merged)<FINAL_PER_TOPIC:
            raise ValueError(f'{variant}/{topic}: {len(merged)} rows; need 300')
        final=merged.sort_values('gpt_final_score',ascending=False).head(min(FINAL_PER_TOPIC,len(merged))).copy()
        final['gpt_final_stance_rank']=pd.NA
    else:
        parts=[]
        for stance in ['pro','anti']:
            sub=merged[merged.stance_label==stance].copy()
            chosen=sub.sort_values('gpt_final_score',ascending=False).head(min(FINAL_PER_STANCE,len(sub))).copy()
            chosen['gpt_final_stance_rank']=range(1,len(chosen)+1)
            parts.append(chosen)
        final=pd.concat(parts,ignore_index=True)
    final=final.sample(frac=1.0,random_state=42).reset_index(drop=True)
    final['topic_gpt_final']=topic
    final['gpt_final_rank_overall']=range(1,len(final)+1)
    return final

def finalize_a4(df,topic):
    parts=[]
    for stance,seed in [('pro',RANDOM_SEED),('anti',RANDOM_SEED+1)]:
        sub=df[df.stance_label==stance]
        if RUN_MODE=='FULL' and len(sub)<FINAL_PER_STANCE:
            raise ValueError(f'A4/{topic}: only {len(sub)} {stance} rows')
        chosen=sub.sample(n=min(FINAL_PER_STANCE,len(sub)),replace=False,random_state=seed).copy()
        chosen['gpt_final_stance_rank']=range(1,len(chosen)+1)
        parts.append(chosen)
    final=pd.concat(parts,ignore_index=True).sample(frac=1.0,random_state=RANDOM_SEED).reset_index(drop=True)
    final['selection_stage_step4']='random_no_llm_refinement'
    final['topic_gpt_final']=topic
    final['gpt_final_rank_overall']=range(1,len(final)+1)
    return final

In [ ]:
# Run selected variants and topics. Existing completed outputs are reused only if the input/config digest matches.
results={}
for variant in RUN_VARIANTS:
    for topic in TOPICS:
        print('\n===',variant,topic,'===')
        df=read_input(variant,topic)
        digest=fingerprint(df,variant,topic)
        outdir=OUTPUT_ROOT/variant;outdir.mkdir(parents=True,exist_ok=True)
        final_path=outdir/f'{topic}_final_{variant}.csv'
        scores_path=outdir/f'{topic}_gpt_scores_all_{variant}.csv'
        manifest_path=outdir/f'{topic}_manifest.json'
        if final_path.exists() and manifest_path.exists():
            with open(manifest_path,encoding='utf-8') as f:manifest=json.load(f)
            if manifest.get('digest')==digest and manifest.get('completed'):
                print('RESUME: completed file already exists; skipping:',final_path)
                results[variant,topic]=pd.read_csv(final_path,encoding='utf-8-sig')
                continue
            raise ValueError(f'Existing output has different digest: {final_path}')
        if variant=='A4':
            final=finalize_a4(df,topic)
        else:
            with_ids,items=gpt_items_with_checkpoints(df,variant,topic,digest)
            merged=merge_scores(with_ids,items)
            merged.to_csv(scores_path,index=False,encoding='utf-8-sig')
            final=finalize_gpt(merged,variant,topic)
        final.to_csv(final_path,index=False,encoding='utf-8-sig')
        atomic_json(manifest_path,{'variant':variant,'topic':topic,'mode':RUN_MODE,'digest':digest,
                                   'input_rows':len(df),'final_rows':len(final),'completed':True,
                                   'model':None if variant=='A4' else AZURE_DEPLOYMENT_NAME,
                                   'seed':RANDOM_SEED,'batch_size':BATCH_SIZE})
        print('Saved',final_path,'rows',len(final))
        if 'stance_label' in final:print(final.stance_label.value_counts(dropna=False).to_dict())
        results[variant,topic]=final
for variant in RUN_VARIANTS:
    combined=pd.concat([results[variant,t] for t in TOPICS],ignore_index=True)
    path=OUTPUT_ROOT/variant/f'all_topics_final_{variant}.csv'
    combined.to_csv(path,index=False,encoding='utf-8-sig')
    print('Combined:',path,len(combined))

## Validation notes

- `FULL` A4: 300 rows per topic (150/150), with no GPT requests or quality-score computation.
- `FULL` A1/A3: original GPT prompt and six-score formula, 150 per stance **provided enough rows exist**. A1 may have fewer than 150 per stance after random 8,000 retrieval; the notebook raises an explicit error instead of fabricating examples.
- `FULL` A2: 300 highest scoring overall; stance quotas are intentionally absent. Topic irrelevance is addressed only by the adapted prompt; it is not equivalent to original stance filtering.
- The baseline postprocessing intentionally retains its original behavior for duplicate/missing GPT IDs (left merge and default scores); this notebook does not silently deduplicate or retry individual IDs. A checkpoint stores the raw batch response.
- To test checkpoint resume, interrupt between two batches, rerun the same configuration, and confirm completed batches are skipped. Checkpoints are persisted in Drive, segregated by TEST/FULL.
- `TEST` uses a small prefix; final TEST row counts are smaller than 300 by design.

In [ ]:
# QA: check output counts and score fields for the variants just run.
qa=[]
for variant in RUN_VARIANTS:
    for topic in TOPICS:
        f=OUTPUT_ROOT/variant/f'{topic}_final_{variant}.csv'
        if not f.exists():
            continue
        d=pd.read_csv(f,encoding='utf-8-sig')
        counts=d['stance_label'].value_counts().to_dict() if 'stance_label' in d else {}
        qa.append({'variant':variant,'topic':topic,'rows':len(d),
                   'pro':counts.get('pro',None),'anti':counts.get('anti',None),
                   'duplicate_comment_id':int(d['comment_id'].duplicated().sum()) if 'comment_id' in d else None,
                   'has_gpt_final_score':'gpt_final_score' in d.columns,
                   'has_final_rank':'gpt_final_rank_overall' in d.columns})
qa_df=pd.DataFrame(qa)
display(qa_df)
qa_path=OUTPUT_ROOT/'step4_summary.csv'
qa_df.to_csv(qa_path,index=False,encoding='utf-8-sig')
print('Saved:',qa_path)